In [4]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# =====================================================
# ASSIGNMENT DATA GENERATOR (DO NOT MODIFY)
# =====================================================

np.random.seed(42)

n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[
        [0.09, 0.02, 0.01],
        [0.02, 0.06, 0.03],
        [0.01, 0.03, 0.05]
    ],
    size=n_samples
)

# Hidden drift
base_data[4000:, 0] += 0.015
base_data[4000:, 2] -= 0.010

df_strain = pd.DataFrame(
    base_data,
    columns=[
        "Strain_Ch1",
        "Strain_Ch2",
        "Strain_Ch3"
    ]
)

# =====================================================
# MANOVA IMPLEMENTATION
# =====================================================

def verify_first_moment_homogeneity(df, g_chunks=5):

    p = df.shape[1]
    N = len(df)

    chunks = np.array_split(df.values, g_chunks)

    global_mean = df.values.mean(axis=0)

    W = np.zeros((p, p))
    B = np.zeros((p, p))

    for chunk in chunks:

        chunk_mean = chunk.mean(axis=0)

        centered = chunk - chunk_mean

        W += centered.T @ centered

        diff = (
            chunk_mean - global_mean
        ).reshape(-1, 1)

        B += len(chunk) * (diff @ diff.T)

    wilks_lambda = np.linalg.det(W) / np.linalg.det(W + B)

    chi_square = (
        -(N - 1 - (p + g_chunks) / 2)
        * np.log(wilks_lambda)
    )

    df_chi = p * (g_chunks - 1)

    p_value = 1 - stats.chi2.cdf(
        chi_square,
        df_chi
    )

    return {
        "Wilks Lambda": wilks_lambda,
        "Bartlett Chi-Square": chi_square,
        "Degrees of Freedom": df_chi,
        "p-value": p_value
    }

# =====================================================
# RUN TEST
# =====================================================

results = verify_first_moment_homogeneity(
    df_strain,
    g_chunks=5
)

print("\nRESULTS")
print("=" * 50)

for k, v in results.items():
    print(f"{k}: {v}")

print("\nCONCLUSION")
print("=" * 50)

alpha = 0.05

if results["p-value"] < alpha:
    print("Reject H0")
    print("Baseline center shifted over time.")
    print("First moment is NOT homogeneous.")
else:
    print("Fail to reject H0")
    print("No significant mean shift detected.")
    print("First moment is homogeneous.")


RESULTS
Wilks Lambda: 0.9904519796929591
Bartlett Chi-Square: 47.9215049961488
Degrees of Freedom: 12
p-value: 3.2255259508895406e-06

CONCLUSION
Reject H0
Baseline center shifted over time.
First moment is NOT homogeneous.


In [3]:

# Install packages if needed
!pip -q install factor_analyzer plotly

import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer, Rotator

# Synthetic asset data

np.random.seed(42)
n_samples = 2500

f1 = np.random.normal(0,1,n_samples)
f2 = np.random.normal(0,1,n_samples)

s1 = 0.85*f1 + 0.10*f2 + np.random.normal(0,0.3,n_samples)
s2 = 0.80*f1 + 0.15*f2 + np.random.normal(0,0.35,n_samples)
s3 = 0.12*f1 + 0.90*f2 + np.random.normal(0,0.25,n_samples)
s4 = 0.02*f1 + 0.05*f2 + np.random.normal(0,1.40,n_samples)

df_asset = pd.DataFrame(
    np.vstack([s1,s2,s3,s4]).T,
    columns=['Sensor_1','Sensor_2','Sensor_3','Sensor_4']
)

df_asset.head()

# MANOVA Function

def verify_first_moment_homogeneity(df,g_chunks=5):

    p=df.shape[1]
    N=len(df)

    chunks=np.array_split(df.values,g_chunks)

    global_mean=df.values.mean(axis=0)

    W=np.zeros((p,p))
    B=np.zeros((p,p))

    for chunk in chunks:

        m=chunk.mean(axis=0)

        Xc=chunk-m

        W+=Xc.T@Xc

        d=(m-global_mean).reshape(-1,1)

        B+=len(chunk)*(d@d.T)

    wilks=np.linalg.det(W)/np.linalg.det(W+B)

    chi2=-(N-1-(p+g_chunks)/2)*np.log(wilks)

    df_chi=p*(g_chunks-1)

    pval=1-stats.chi2.cdf(chi2,df_chi)

    return wilks,chi2,pval

# PCA Analysis

X=StandardScaler().fit_transform(df_asset)

cov=np.cov(X,rowvar=False,ddof=1)

eigvals,eigvecs=np.linalg.eigh(cov)

idx=np.argsort(eigvals)[::-1]

eigvals=eigvals[idx]
eigvecs=eigvecs[:,idx]

explained=eigvals/eigvals.sum()*100
cum=np.cumsum(explained)
residual=100-cum

t2_mean=[]
spe_mean=[]

for r in range(1,len(eigvals)+1):

    P=eigvecs[:,:r]

    scores=X@P

    T2=np.sum((scores**2)/eigvals[:r],axis=1)

    Xhat=scores@P.T

    E=X-Xhat

    SPE=np.sum(E**2,axis=1)

    t2_mean.append(T2.mean())
    spe_mean.append(SPE.mean())

# PCA Dashboard

fig = make_subplots(
rows=2,
cols=3,
subplot_titles=[
'Loadings Heatmap',
'Eigenvalues',
'Explained vs Cumulative',
'Residual Variance',
'Mean T²',
'Mean SPE'
])

fig.add_trace(
go.Heatmap(
z=eigvecs,
x=[f'PC{i+1}' for i in range(4)],
y=df_asset.columns
),
row=1,col=1)

fig.add_trace(
go.Bar(
x=[f'PC{i+1}' for i in range(4)],
y=eigvals
),
row=1,col=2)

fig.add_trace(
go.Bar(
x=[f'PC{i+1}' for i in range(4)],
y=explained,
name='Explained %'
),
row=1,col=3)

fig.add_trace(
go.Scatter(
x=[f'PC{i+1}' for i in range(4)],
y=cum,
mode='lines+markers',
name='Cumulative %'
),
row=1,col=3)

fig.add_trace(
go.Bar(
x=[f'PC{i+1}' for i in range(4)],
y=residual
),
row=2,col=1)

fig.add_trace(
go.Scatter(
x=list(range(1,5)),
y=t2_mean,
mode='lines+markers'
),
row=2,col=2)

fig.add_trace(
go.Scatter(
x=list(range(1,5)),
y=spe_mean,
mode='lines+markers'
),
row=2,col=3)

fig.update_layout(
height=700,
width=1400,
template='plotly_white',
title='PCA Optimization Dashboard'
)

fig.show()

# Factor Analysis

fa=FactorAnalyzer(
n_factors=2,
rotation=None,
method='ml'
)

fa.fit(X)

loadings=fa.loadings_

rotator=Rotator(method='varimax')

loadings=rotator.fit_transform(loadings)

communalities=np.sum(loadings**2,axis=1)

uniqueness=1-communalities

scores=X@loadings@np.linalg.inv(
loadings.T@loadings+np.eye(2)
)

factor_var=np.var(scores,axis=0,ddof=1)

# Factor Analysis Dashboard

fig = make_subplots(
rows=2,
cols=2,
horizontal_spacing=0.24,
vertical_spacing=0.28,
subplot_titles=[
'Structural Loadings',
'Communality vs Uniqueness',
'Noise Floor',
'Latent Variance'
])

fig.add_trace(
go.Heatmap(
z=np.abs(loadings),
x=['Factor1','Factor2'],
y=df_asset.columns,
colorscale='YlOrRd'
),
row=1,col=1)

fig.add_trace(
go.Bar(
y=df_asset.columns,
x=communalities*100,
orientation='h',
name='Communality'
),
row=1,col=2)

fig.add_trace(
go.Bar(
y=df_asset.columns,
x=uniqueness*100,
orientation='h',
name='Uniqueness'
),
row=1,col=2)

fig.add_trace(
go.Scatter(
x=df_asset.columns,
y=uniqueness,
mode='lines+markers'
),
row=2,col=1)

fig.add_trace(
go.Bar(
x=['Factor1','Factor2'],
y=factor_var
),
row=2,col=2)

fig.update_layout(
width=1250,
height=750,
template='plotly_white',
barmode='stack',
title='Factor Analysis Diagnostic Dashboard'
)

fig.show()


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



Q1

In [9]:
# Question 1: Total Variance Illusion (PCA vs FA)

print("QUESTION 1: Total Variance Illusion (PCA vs FA)\n")

# Given value from prompt
sensor_4_uniqueness = 0.98  # > 98%

print("Sensor_4 Uniqueness (φ²):", sensor_4_uniqueness, "\n")

print("1. WHAT DOES UNIQUENESS ≈ 1 MEAN?")
print("- Sensor_4 is almost entirely UNIQUE variance.")
print("- It is NOT explained by any latent common factor.")
print("- Physically, this means:")
print("  → Sensor_4 is dominated by local noise / sensor-specific disturbance")
print("  → It is weakly coupled to system-wide dynamics\n")

print("2. WHY THIS CREATES A PCA ILLUSION:")
print("- PCA does NOT separate common vs unique variance.")
print("- It maximizes TOTAL variance (signal + noise together).")
print("- So the huge noise in Sensor_4 inflates eigenvalues of early PCs.")
print("- This makes PCA falsely believe the system has stronger structure than it actually does.\n")

print("3. HOW SENSOR_4 'TRICKS' PCA:")
print("- Local high-variance noise is treated as meaningful direction.")
print("- PCA loads this noise into PC1/PC2.")
print("- Result:")
print("  → inflated explained variance (e.g., 45% + 30%)")
print("  → but this variance is not physically meaningful structure\n")

print("4. WHY THIS IS DANGEROUS IN MONITORING:")
print("- System appears more stable/structured than reality.")
print("- True latent faults get hidden under noise-dominated components.")
print("- Leads to incorrect anomaly detection thresholds.\n")

print("FINAL CONCLUSION:")
print("Sensor_4 with φ² ≈ 1 is essentially a noise-dominated channel that distorts PCA’s variance structure,")
print("creating a misleading impression of strong system organization.")

QUESTION 1: Total Variance Illusion (PCA vs FA)

Sensor_4 Uniqueness (φ²): 0.98 

1. WHAT DOES UNIQUENESS ≈ 1 MEAN?
- Sensor_4 is almost entirely UNIQUE variance.
- It is NOT explained by any latent common factor.
- Physically, this means:
  → Sensor_4 is dominated by local noise / sensor-specific disturbance
  → It is weakly coupled to system-wide dynamics

2. WHY THIS CREATES A PCA ILLUSION:
- PCA does NOT separate common vs unique variance.
- It maximizes TOTAL variance (signal + noise together).
- So the huge noise in Sensor_4 inflates eigenvalues of early PCs.
- This makes PCA falsely believe the system has stronger structure than it actually does.

3. HOW SENSOR_4 'TRICKS' PCA:
- Local high-variance noise is treated as meaningful direction.
- PCA loads this noise into PC1/PC2.
- Result:
  → inflated explained variance (e.g., 45% + 30%)
  → but this variance is not physically meaningful structure

4. WHY THIS IS DANGEROUS IN MONITORING:
- System appears more stable/structured than

Q2

In [10]:
# Question 2: Varimax Rotation vs PCA Eigenvectors

print("QUESTION 2: Decoupling Structure via Varimax Rotation\n")

print("1. PCA BEHAVIOR:")
print("- PCA eigenvectors follow strict variance hierarchy (λ1 > λ2 > ...).")
print("- Each component captures MIXED variance across all sensors.")
print("- Loadings are distributed → physically unclear structure.\n")

print("2. PROBLEM WITH PCA STRUCTURE:")
print("- Sensor contributions are blended across multiple PCs.")
print("- No clean separation of physical subsystems.")
print("- Fault interpretation becomes ambiguous.\n")

print("3. VARIMAX ROTATION (FA):")
print("- Rotation does NOT change total explained variance.")
print("- It rotates axes to maximize 'simple structure'.")
print("- Simple structure means:")
print("  → Loadings become either strong (≈1) or weak (≈0)")
print("  → Each sensor aligns strongly with ONE factor only\n")

print("4. WHAT HAPPENS IN YOUR SYSTEM:")
print("- Sensor_1 + Sensor_2 → Factor 1 (shared subsystem)")
print("- Sensor_3 → Factor 2 (isolated subsystem)")
print("- This clean grouping reveals real physical decoupling\n")

print("5. WHY FA IS BETTER FOR TROUBLESHOOTING:")
print("- PCA: failure spreads across multiple components → confusing diagnosis")
print("- FA: failure is localized to a factor → easy root-cause identification")
print("- Operators can directly map factors to machine subsystems\n")

print("FINAL CONCLUSION:")
print("Varimax rotation transforms PCA’s mathematically optimal but mixed structure into a physically interpretable subsystem map,")
print("making FA far superior for diagnosing structural failures in engineering systems.")

QUESTION 2: Decoupling Structure via Varimax Rotation

1. PCA BEHAVIOR:
- PCA eigenvectors follow strict variance hierarchy (λ1 > λ2 > ...).
- Each component captures MIXED variance across all sensors.
- Loadings are distributed → physically unclear structure.

2. PROBLEM WITH PCA STRUCTURE:
- Sensor contributions are blended across multiple PCs.
- No clean separation of physical subsystems.
- Fault interpretation becomes ambiguous.

3. VARIMAX ROTATION (FA):
- Rotation does NOT change total explained variance.
- It rotates axes to maximize 'simple structure'.
- Simple structure means:
  → Loadings become either strong (≈1) or weak (≈0)
  → Each sensor aligns strongly with ONE factor only

4. WHAT HAPPENS IN YOUR SYSTEM:
- Sensor_1 + Sensor_2 → Factor 1 (shared subsystem)
- Sensor_3 → Factor 2 (isolated subsystem)
- This clean grouping reveals real physical decoupling

5. WHY FA IS BETTER FOR TROUBLESHOOTING:
- PCA: failure spreads across multiple components → confusing diagnosis
- FA:

Q3

In [7]:
# Question 3: Determining Subspace Truncation (k) using T² and Q Profiles

print("QUESTION 3: Subspace Truncation using T² and Q Statistic\n")

print("1. BEHAVIOR OF Mean Q Statistic (SPE):")
print("- At k = 1 → Q is high because most system structure is not captured.")
print("- From k = 1 → k = 2 → sharp drop in Q occurs.")
print("  This indicates the model suddenly captures the dominant physical structure.")
print("- At k = 2 → Q reaches a distinct 'flat elbow region'.")
print("- From k = 2 → k = 3 → Q curve becomes nearly flat (no significant improvement).")
print("  This means additional components mostly model noise, not structure.\n")

print("2. WHY k = 2 IS THE TRUE DIMENSION:")
print("- The sharp drop followed by a flat elbow is a classic dimensionality signature.")
print("- It indicates that the system has exactly TWO meaningful latent sources.")
print("- Beyond k = 2, improvements are negligible → only noise is being fitted.\n")

print("3. WHAT IF YOU INCORRECTLY CHOOSE k = 3:")
print("- You force residual noise into the principal subspace.")
print("- This causes 'noise leakage' into PCA model structure.")
print("- Monitoring system becomes less sensitive to real faults.")
print("- False interpretation of noise as weak physical dynamics.\n")

print("FINAL CONCLUSION:")
print("The Q-statistic elbow at k = 2 reveals true system dimensionality,")
print("and selecting k = 3 contaminates the clean physical subspace with noise.")

QUESTION 3: Subspace Truncation using T² and Q Statistic

1. BEHAVIOR OF Mean Q Statistic (SPE):
- At k = 1 → Q is high because most system structure is not captured.
- From k = 1 → k = 2 → sharp drop in Q occurs.
  This indicates the model suddenly captures the dominant physical structure.
- At k = 2 → Q reaches a distinct 'flat elbow region'.
- From k = 2 → k = 3 → Q curve becomes nearly flat (no significant improvement).
  This means additional components mostly model noise, not structure.

2. WHY k = 2 IS THE TRUE DIMENSION:
- The sharp drop followed by a flat elbow is a classic dimensionality signature.
- It indicates that the system has exactly TWO meaningful latent sources.
- Beyond k = 2, improvements are negligible → only noise is being fitted.

3. WHAT IF YOU INCORRECTLY CHOOSE k = 3:
- You force residual noise into the principal subspace.
- This causes 'noise leakage' into PCA model structure.
- Monitoring system becomes less sensitive to real faults.
- False interpretation 

Q4

In [8]:
# Question 4: Operational Trade-offs in System Health Monitoring

print("QUESTION 4: PCA vs FA for System Health Monitoring\n")

print("1. PCA STRATEGY:")
print("- Uses Hotelling's T² for global variation monitoring.")
print("- Uses Q-statistic (SPE) for residual/noise detection.")
print("- Strength: good for detecting global deviations in system behavior.")
print("- Weakness: sensitive to sensor mixing and correlated noise.\n")

print("2. FACTOR ANALYSIS (FA) STRATEGY:")
print("- Monitors rotated latent factor scores directly.")
print("- Each factor corresponds to a physical subsystem.")
print("- Uniqueness (φ²) isolates sensor-specific noise behavior.")
print("- Strength: interpretable and physically localized fault detection.\n")

print("3. ROBUSTNESS TO SENSOR FAILURE (Calibration loss / short circuit):")
print("- If a single sensor fails:")
print("  PCA → failure spreads across multiple principal components.")
print("  FA → failure remains mostly isolated due to high uniqueness structure.\n")

print("4. ROLE OF SENSOR UNIQUENESS (φ²):")
print("- High φ² means sensor behavior is mostly independent noise.")
print("- Faults in one sensor do NOT strongly distort latent factors.")
print("- This prevents system-wide contamination of monitoring metrics.\n")

print("FINAL VERDICT:")
print("- FA Strategy is more robust to single sensor failure or electrical faults.")
print("- Reason: rotated factors + uniqueness structure localize disturbances.")
print("- PCA is more sensitive to sensor coupling, causing global distortion of T²/Q space.")

QUESTION 4: PCA vs FA for System Health Monitoring

1. PCA STRATEGY:
- Uses Hotelling's T² for global variation monitoring.
- Uses Q-statistic (SPE) for residual/noise detection.
- Strength: good for detecting global deviations in system behavior.
- Weakness: sensitive to sensor mixing and correlated noise.

2. FACTOR ANALYSIS (FA) STRATEGY:
- Monitors rotated latent factor scores directly.
- Each factor corresponds to a physical subsystem.
- Uniqueness (φ²) isolates sensor-specific noise behavior.
- Strength: interpretable and physically localized fault detection.

3. ROBUSTNESS TO SENSOR FAILURE (Calibration loss / short circuit):
- If a single sensor fails:
  PCA → failure spreads across multiple principal components.
  FA → failure remains mostly isolated due to high uniqueness structure.

4. ROLE OF SENSOR UNIQUENESS (φ²):
- High φ² means sensor behavior is mostly independent noise.
- Faults in one sensor do NOT strongly distort latent factors.
- This prevents system-wide contamin